In [18]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [21]:
load_dotenv()

True

In [22]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [23]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [24]:
def create_outline(state:BlogState) -> BlogState:
    # fetch title

     title = state['title']

     #call llm to generate outline
     
     prompt = f'Generate a detailed outline for a blog on topic {title}'
     outline = model.invoke(prompt).content
     
     # update state
     state['outline'] = outline

     return state

In [10]:
def create_blog(state:BlogState)-> BlogState:

    # fetch outline and title
    title = state['title']
    outline = state['outline']

    # call llm to generate blog

    prompt = f'Write a detailed blog on the topic - {title} using the following outline \n {outline} '
    content = model.invoke(prompt).content
    # update state
    state['content'] = content

    return state

In [25]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow = graph.compile()


In [26]:
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here\'s a detailed outline for a blog post on the "Rise of AI in India," designed to be comprehensive, engaging, and informative for a broad audience.\n\n---\n\n## Blog Title Options:\n\n*   **The AI Ascent: How India is Redefining the Future with Artificial Intelligence**\n*   **India\'s AI Revolution: Unpacking the Rise, Impact, and Future of AI Across the Nation**\n*   **From Code to Catalyst: The Unstoppable Rise of AI in India**\n*   **Beyond the Hype: Understanding India\'s Unique Journey in the AI Landscape**\n\n---\n\n## Blog Outline: The AI Ascent: How India is Redefining the Future with Artificial Intelligence\n\n**Target Audience:** Tech enthusiasts, business leaders, policymakers, students, general public interested in India\'s technological advancements.\n\n**Tone:** Optimistic, informative, analytical, forward-looking.\n\n**Keywords:** AI in India, Indian AI ecosystem, AI startups India, Digital India, AI for good, Indian tech 